# Joint deconvolution of paired molecular assays

Supply `rna.h5ad`, `protein.h5ad` and `feature_links.csv`. Assays share the same spots and spatial
coordinates, but may have different feature panels. `feature_links.csv` has columns `RNA` and
`protein`; each row names corresponding features, with no repeated feature within an assay.
For example, an RNA gene symbol can be paired with the corresponding antibody feature name.

The model learns shared spatial weights and assay-specific profiles. Coupling acts on matched,
scaled features while reconstruction retains each assay's full fitting panel. Spatial TV acts
once on the shared weights. Input RNA uses raw counts; protein uses your prepared nonnegative
measurements. Choose the assay preprocessing and component count for your data.

## Data links

- Paired tonsil RNA/protein measurements, H&E and registration: [10x Visium CytAssist Human Tonsil](https://www.10xgenomics.com/datasets/visium-cytassist-gene-and-protein-expression-library-of-human-tonsil-with-add-on-antibodies-h-e-6-5-mm-ffpe-2-standard).
- Optional pathology annotations: [DGAT data release](https://github.com/osmanbeyoglulab/DGAT), `Tonsil_AddOns_ADT.h5ad`, `obs["pathology"]`.

## Example figure

Archived tonsil example: eleven joint components on the Visium hexagonal lattice.
Each map has its own labelled 99th-percentile display limit; compare values using its colorbar.

![Eleven joint RNA-protein component maps in human tonsil](../docs/source/_static/figures/tonsil_joint_structure.png)


## 1. Configure your input

Install the notebook dependencies, then edit `DATA_DIR`, the input filenames and `K` below.
Supply your own `.h5ad` files with unique spot and feature IDs. RNA fitting uses raw counts;
set `COUNTS_LAYER="counts"` when they are stored in a layer. Protein inputs should contain
your prepared nonnegative assay measurements. Each fitted spot needs positive total signal.

Running the fitting cells trains a new model. `DEVICE` accepts `"auto"`, `"cpu"` or `"cuda"`.
The study's reference-free and joint analyses used GPUs. Inspect the returned diagnostics
to determine whether your fit converged. Large inputs require enough memory for dense matrices.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from IPython.display import display
from tutorials.common import correlation, align, spatial, pies, legend, save_figure, save_table
from spDecon.reproducibility import set_seed

DATA_DIR = Path("/path/to/your/data")
COUNTS_LAYER = None
DEVICE = "auto"
VERBOSE = True
MAX_ITER = 20000
set_seed(1, deterministic=True)

K = 11  # Choose the component number for your data.
ETA = 10.
SPATIAL_WEIGHT = .05


## 2. Read paired assays and feature links


In [ ]:
rna_adata = ad.read_h5ad(DATA_DIR / "rna.h5ad")
protein_adata = ad.read_h5ad(DATA_DIR / "protein.h5ad")
if (not rna_adata.obs_names.is_unique or not protein_adata.obs_names.is_unique
        or set(rna_adata.obs_names) != set(protein_adata.obs_names)):
    raise ValueError("Paired assays need identical, unique spot IDs")
protein_adata = protein_adata[rna_adata.obs_names].copy()
xy = np.asarray(rna_adata.obsm["spatial"])
if "spatial" in protein_adata.obsm:
    np.testing.assert_allclose(protein_adata.obsm["spatial"], xy)
sections = rna_adata.obs["section_id"].to_numpy() if "section_id" in rna_adata.obs else np.repeat("section_1", len(xy))
assays = {"RNA": rna_adata.to_df(layer=COUNTS_LAYER), "protein": protein_adata.to_df()}
print({name: values.shape for name, values in assays.items()})

link_table = pd.read_csv(DATA_DIR / "feature_links.csv", dtype=str)
links = {name: link_table[name].tolist() for name in assays}
display(link_table)


## 3. Fit the shared spatial weights


In [ ]:
from spDecon.multiomics import fit_multiomics
result = fit_multiomics(assays, K, links=links, coordinates=xy, sections=sections,
    eta=ETA, spatial_weight=SPATIAL_WEIGHT, random_state=1, device=DEVICE,
    max_iter=MAX_ITER, verbose=VERBOSE)
p, b = result.proportions, result.profiles
components = [f"C{k+1}" for k in range(K)]
display(result.diagnostics)


## 4. Display shared components

Use physical coordinates for your platform. If your metadata contains Visium array row and
column indices, convert them to physical or hexagonal display coordinates before using them
here. Component maps use the same 0–1 weight scale.


In [ ]:
fig, axes = plt.subplots(int(np.ceil(p.shape[1]/4)), 4,
                         figsize=(12, 3*int(np.ceil(p.shape[1]/4))), squeeze=False)
for k, ax in enumerate(axes.ravel()):
    if k < p.shape[1]:
        art = spatial(ax, xy, p[:,k], components[k], size=10)
    else:
        ax.axis("off")
fig.colorbar(art, ax=axes, label="Molecular contribution", shrink=.6)
save_figure(fig, "multiomics_components")


## 5. Save weights and assay-specific profiles


In [ ]:
out = ROOT / "outputs/multiomics"
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame(p, index=rna_adata.obs_names, columns=components).to_csv(out / "shared_weights.csv")
offsets = result.extra["feature_offsets"]
for i, name in enumerate(result.extra["assays"]):
    profile = b[:, offsets[i]:offsets[i+1]]
    pd.DataFrame(profile, index=components, columns=assays[name].columns).to_csv(out / f"{name}_profiles.csv")
display(save_table(pd.DataFrame({"component": components, "mean_weight": p.mean(axis=0)}), "multiomics_summary"))
